In [18]:
from dotenv import load_dotenv
from pathlib import Path
import sys
import os

# Walk up until we find the project root (folder with the .env)
current_path = Path().resolve()
for parent in [current_path] + list(current_path.parents):
    if (parent / ".env").exists():
        load_dotenv(parent / ".env")
        project_root = os.getenv("PROJECT_ROOT")
        print(project_root)
        sys.path.append(project_root)     
        break


%load_ext autoreload
%autoreload 2

C:\Users\Admin\Desktop\Emmanuel\beluga-call-pipeline\
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [19]:


import torch
import matplotlib.pyplot as plt
from datetime import timedelta

import numpy as np

import matplotlib.pyplot as plt

import time
from tqdm import tqdm
from datetime import datetime
import os
from data_preprocessing.spectrogram.spectrogram_generator import SPECT_GENERATOR, HYDROPHONE_SENSITIVITY
import pandas as pd
from models.quant_mobilenet import load_mobilenet_v3_quant_from_file
from data_preprocessing.spectrogram.spectrogram_generator import SpectrogramGenerator
from pipeline import classify, run_pipeline_batched_long_spects,get_audio_start_time, get_hydrophone_model

from vizualization import plot_pipeline_outputs_on_spects, analyze_pipeline_performance

from models.utils import get_best_device
%load_ext autoreload
%autoreload 2

# Set pandas display options to show full dataframe
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None) 
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:

model = load_mobilenet_v3_quant_from_file("../models/weights/mobile_net_8_layers_qat.pt", n_layers=8)

device = get_best_device()
model.to(device)
model.eval()

#spectrogram generator the model was trained on
spect_generator = SpectrogramGenerator(
        n_fft=2048,
        hop_length=1024,
        n_mels=64,
        fmin=200,
        sample_rate=192000, 
    )

C:\Users\Admin\Desktop\Emmanuel\beluga-call-pipeline\models\quant_mobilenet.py:85: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(path, map_location="

### Loading the file we want to process



In [21]:
def translate_to_raven(results_df, sample_rate):
    # Create new dataframe with Raven format columns
    raven_df = pd.DataFrame()
    raven_df["Selection"] = [x + 1 for x in results_df.index.tolist()]
    raven_df["View"] = "Spectrogram 1"
    raven_df["Channel"] = 1
    # Begin Time is seconds since start
    raven_df["Begin Time (s)"] = results_df["seconds_since_file_start"]
    
    # End Time is Begin Time + 1 second window
    raven_df["End Time (s)"] = results_df["seconds_since_file_start"] + 1.0
    
    # Low Freq is 0 Hz
    raven_df["Low Freq (Hz)"] = 0.0
    
    # High Freq is Nyquist frequency (sample_rate/2)
    raven_df["High Freq (Hz)"] = sample_rate / 2

    raven_df["ECHO"] = results_df["ECHO"]
    raven_df["HFPC"] = results_df["HFPC"]
    raven_df["BBPC"] = results_df["CC"]
    raven_df["Whistle"] = results_df["Whistle"]

    raven_df["GROUNDTRUTH"] = ""
    raven_df["DETAILS"] = ""
    raven_df["Notes"] = ""
    
    return raven_df
    

In [22]:
def clip_and_save_audio(long_audio, sample_rate, start_s, end_s, output_filename, dir="./"):
    """
    Clip a segment from audio and save it to a file.
    
    Parameters:
    -----------
    long_audio : np.ndarray
        The full audio array (float32 from librosa)
    sample_rate : int
        Sample rate of the audio (Hz)
    start_s : float
        Start time in seconds
    end_s : float
        End time in seconds
    output_filename : str
        Filename to save the clipped audio file (should end with .wav)
    dir : str
        Directory to save the file (default: "./")
    
    Returns:
    --------
    np.ndarray
        The clipped audio segment
    """
    import soundfile as sf
    import os
    
    # Convert times to sample indices
    start_sample = int(start_s * sample_rate)
    end_sample = int(end_s * sample_rate)
    
    # Clip the audio
    clipped_audio = long_audio[start_sample:end_sample]
    
    # Ensure directory exists
    output_dir = os.path.abspath(dir)
    os.makedirs(output_dir, exist_ok=True)
    filepath = os.path.join(output_dir, output_filename)
    
    # Save using soundfile - handles float32 properly for all audio software
    sf.write(filepath, clipped_audio, sample_rate)
    
    duration = end_s - start_s
    print(f"Saved {duration:.2f}s audio clip to: {filepath}")
    print(f"Clip shape: {clipped_audio.shape}, Sample rate: {sample_rate} Hz")
    
    return clipped_audio

In [23]:
def create_snippet_df( raven_df, filename, start_s, end_s):
    audio_start_time = get_audio_start_time(filename)
    hydrophone_model = get_hydrophone_model(filename)

    snippet_df = raven_df[
            (raven_df["Begin Time (s)"] >= start_s) & 
            (raven_df["Begin Time (s)"] < end_s)
        ].copy()

    snippet_df["Begin Time (s)"] = snippet_df["Begin Time (s)"] - start_s
    snippet_df["End Time (s)"] = snippet_df["End Time (s)"] - start_s

    snippet_start_time = audio_start_time + pd.to_timedelta(start_s, unit="s")
    # Format snippet_start_time to create the date string in the filename (YYMMDDHHMMSS)
    snippet_date_str = snippet_start_time.strftime("%y%m%d%H%M%S")

    snippet_filename = f"{hydrophone_model}.{snippet_date_str}.snippet.wav"

    snippet_df["original_filename"] = filename
    snippet_df["snippet_filename"] = snippet_filename
    snippet_df["snippet_start_time"] = snippet_start_time
    snippet_df["snippet_start_s"] = start_s
    snippet_df["snippet_end_s"] = end_s
    

    # calls_detected = snippet_df["ECHO"].sum() + snippet_df["HFPC"].sum() + snippet_df["BBPC"].sum() + snippet_df["Whistle"].sum()

    calls_detected = {
        "ECHO": snippet_df["ECHO"].sum(),
        "HFPC": snippet_df["HFPC"].sum(),
        "BBPC": snippet_df["BBPC"].sum(),
        "Whistle": snippet_df["Whistle"].sum()
    }

    return snippet_df, snippet_filename, calls_detected

    

## Exporting the file snippets


6871.220829124106 : 35/36- 38.5
6871.220821132704 : 2-4
6871.220821132704 : 9-12
6871.220829124106 : 33-39

In [40]:
file_of_interest = "6871.220829124106"

site = "RDL"
site_year = "RDL_2022"
wavs_folder = "J:/"

audio_file = file_of_interest+".wav"
audio_file_path = wavs_folder + audio_file


# # print(audio_start_time)
long_audio, sample_rate, original_sr = spect_generator.load_audio(audio_file_path) 

In [41]:
results_df = pd.read_csv(f"./outputs/{site_year}/{file_of_interest}.csv")
raven_df = translate_to_raven(results_df, sample_rate)

In [42]:
raven_df.head(4)

,Selection,View,Channel,Begin Time (s),End Time (s),Low Freq (Hz),High Freq (Hz),ECHO,HFPC,BBPC,Whistle,GROUNDTRUTH,DETAILS,Notes
0,1,Spectrogram 1,1,4.0,5.0,0.0,96000.0,False,False,True,False,,,
1,2,Spectrogram 1,1,5.0,6.0,0.0,96000.0,True,False,False,False,,,
2,3,Spectrogram 1,1,6.0,7.0,0.0,96000.0,False,False,False,False,,,
3,4,Spectrogram 1,1,7.0,8.0,0.0,96000.0,False,False,False,False,,,


In [43]:
# boat_labeling_file = f"{file_of_interest}.Table.1.selections.txt"
# boat_labels = pd.read_csv(f"{labels_folder}/{site_year}/{boat_labeling_file}", sep="\t")
# boat_labels["Begin Time (s)"] = boat_labels["Begin Time (s)"].round(0).astype(int)
# boat_labels["End Time (s)"] = boat_labels["End Time (s)"].round(0).astype(int)

In [44]:
snippet_dir = f"../data/evaluation_snippets/irene/{site_year}/"

In [45]:

# passage_id = 1
# passage_start_s = boat_labels.loc[boat_labels["Selection"] == passage_id, "Begin Time (s)"].values[0]
# passage_end_s = boat_labels.loc[boat_labels["Selection"] == passage_id, "End Time (s)"].values[0]
# context = boat_labels.loc[boat_labels["Selection"] == 1, "Type"].values[0]
# print(f"Passage {passage_id}: {passage_start_s}s to {passage_end_s}s, context: {context}, length: {passage_end_s - passage_start_s}s")

In [46]:

start_s = 33*60 
end_s = int(39*60)

passage_df, snippet_filename, calls_detected = create_snippet_df(raven_df, audio_file, start_s, end_s)
passage_df["Boat"] = 1
print(f"Calls detected in passage: {calls_detected}")

clip_and_save_audio(long_audio, sample_rate, start_s, end_s, snippet_filename, snippet_dir)

labels_filename = snippet_filename.replace(".wav", ".selections.txt")
passage_df.to_csv(snippet_dir + labels_filename, sep="\t", index=False)



Calls detected in passage: {'ECHO': np.int64(69), 'HFPC': np.int64(7), 'BBPC': np.int64(2), 'Whistle': np.int64(0)}
Saved 360.00s audio clip to: c:\Users\Admin\Desktop\Emmanuel\beluga-call-pipeline\data\evaluation_snippets\irene\RDL_2022\6871.220829131406.snippet.wav
Clip shape: (69120000,), Sample rate: 192000 Hz


Interrupted — exiting processing loop early.


# Merging everything

In [11]:
labels_root = "../data/Labels_boats/"
out_fp = "../data/Labels_boats/labels_boats_merged.csv"
dfs = []
master_columns = None

for site_year in os.listdir(labels_root):
    site_path = os.path.join(labels_root, site_year)
    if not os.path.isdir(site_path):
        continue

    for fname in os.listdir(site_path):
        if not fname.endswith(".txt"):
            continue

        file_path = os.path.join(site_path, fname)

        # Read file raw
        df = pd.read_csv(file_path, sep="\t")

        # First file defines the canonical schema
        if master_columns is None:
            master_columns = df.columns.tolist()
        else:
            # Drop repeated header rows accidentally parsed as data
            df = df[df[df.columns[0]] != master_columns[0]]

        # Add metadata
        df["site_year"] = site_year
        df["source_label_file"] = fname

        dfs.append(df)

# Merge everything
labels_boats_merged = pd.concat(dfs, ignore_index=True)

labels_boats_merged["Begin Time (s)"] = labels_boats_merged["Begin Time (s)"].round(0).astype(int)
labels_boats_merged["End Time (s)"] = labels_boats_merged["End Time (s)"].round(0).astype(int)
labels_boats_merged["duration_s"] = labels_boats_merged["End Time (s)"] - labels_boats_merged["Begin Time (s)"]
labels_boats_merged.loc[labels_boats_merged["Type"] == "Empty", "Type"] = "Ambient"

labels_boats_merged = labels_boats_merged.dropna()
# # Save clean CSV
# os.makedirs(os.path.dirname(out_fp), exist_ok=True)
# labels_boats_merged.to_csv(out_fp, index=False)

# print(f"Merged dataset saved to: {out_fp}")
# print(f"Total rows: {len(labels_boats_merged)}")

In [12]:
labels_boats_merged.head(5)
labels_boats_merged["duration_s"].describe()

count     154.000000
mean      763.012987
std       520.903890
min        98.000000
25%       384.500000
50%       682.500000
75%       990.000000
max      3924.000000
Name: duration_s, dtype: float64

In [13]:
labels_boats_merged["Type"].value_counts(dropna=False)

Type
Boat       77
Ambient    77
Name: count, dtype: int64

In [ ]:
# labels_boats_merged = labels_boats_merged[labels_boats_merged["Begin File"] == "201359382.210714075958.wav"]

In [14]:
def create_passage_snippets(passage_start_s, passage_end_s, min_duration_s=240, snippet_duration_s=120, max_snippets=3):
    """
    Create snippet time ranges from a passage based on its duration.
    
    Args:
        passage_start_s: Start time of the passage in seconds
        passage_end_s: End time of the passage in seconds
        min_duration_s: Minimum duration to split into snippets (default 4 minutes = 240 seconds)
        snippet_duration_s: Duration of each snippet (default 2 minutes = 120 seconds)
        max_snippets: Maximum number of snippets to create (default 5)
    
    Returns:
        List of tuples (start_s, end_s) representing snippet time ranges
    
    Examples:
        - 3 minutes (180s): returns whole passage as 1 snippet
        - 5 minutes (300s): returns 2 snippets of 2 minutes each
        - 8 minutes (480s): returns 3 snippets of 2 minutes each, evenly distributed
        - 50 minutes (3000s): returns 5 snippets of 2 minutes each, evenly distributed
    """
    duration_s = passage_end_s - passage_start_s
    
    # If duration is shorter than minimum, return the whole passage
    if duration_s < min_duration_s:
        return [(passage_start_s, passage_end_s)]
    
    # Calculate number of snippets (capped at max_snippets)
    num_snippets = min(max_snippets, int(duration_s // snippet_duration_s))
    
    # If we only have 1 snippet, return the whole passage
    if num_snippets <= 1:
        end_s = passage_start_s + snippet_duration_s
        return [(passage_start_s, min(end_s, passage_end_s))]
    
    # Distribute snippets evenly across the passage
    # Available range for snippet start positions
    available_range = duration_s - snippet_duration_s
    
    snippets = []
    for i in range(num_snippets):
        # Calculate start position: evenly distributed from beginning to end
        start_offset = (i / (num_snippets - 1)) * available_range
        start_s = passage_start_s + start_offset
        end_s = start_s + snippet_duration_s

        start_s = round(start_s)
        end_s = round(end_s)
        snippets.append((start_s, end_s))
    
    return snippets

In [15]:
create_passage_snippets(0, 800, max_snippets=3)

[(0, 120), (340, 460), (680, 800)]

In [16]:
labels_boats_merged.head(5)

,Selection,View,Channel,Begin Time (s),End Time (s),High Freq (Hz),Low Freq (Hz),Begin Date Time,Delta Time (s),Begin File,Type,site_year,source_label_file,duration_s
0,1,Spectrogram 1,1,1035,1551,61166.144,0.0,2017/7/24 9:47:16.9044,515.9344,201359382.170724093002.wav,Boat,BSM_2017,201359382.170724093002.Table.1.selections.txt,516
1,2,Spectrogram 1,1,0,912,144000.000,0.0,2017/7/24 9:30:02.0000,912.1102,201359382.170724093002.wav,Ambient,BSM_2017,201359382.170724093002.Table.1.selections.txt,912
2,1,Spectrogram 1,1,161,686,45113.900,0.0,2017/7/24 10:32:43.3788,525.1063,201359382.170724103002.wav,Boat,BSM_2017,201359382.170724103002.Table.1.selections.txt,525
4,1,Spectrogram 1,1,1993,2311,30039.700,0.0,2017/7/24 12:03:15.3663,318.0901,201359382.170724113002.wav,Boat,BSM_2017,201359382.170724113002.Table.1.selections.txt,318
5,2,Spectrogram 1,1,1209,1725,61166.100,0.0,2017/7/24 11:50:11.3403,515.7655,201359382.170724113002.wav,Boat,BSM_2017,201359382.170724113002.Table.1.selections.txt,516


In [34]:
snippet_dir = f"../data/evaluation_snippets/v2/"

# wavs_folder_dict = {
#     "KAM_2020": "G:/KAM/WAV files/",
#     "BSM_2017": "D:/BSM 2017/",
#     "CAC_2021": "/Users/emmanuel/Documents/belugas/beluga-call-pipeline/data/CAC_2021_Labels_Ship/"
# }
wavs_folder_dict = {
    "KAM_2020": "G:/KAM/WAV files/",
    "BSM_2017": "D:/BSM 2017/",
    "CAC_2021": "G:/CAC_2021/"
}

# snippets_df = pd.DataFrame(columns=["site_year", "snippet_filename", "pipeline_output_file", "source_wav_file", "source_label_file", "context", "source_label_passage_id", "original_start_s", "original_end_s", "snippet_passage_timing", "n_calls_detected", "labeled"])
snippets_list = []

for (site_year, audio_file), group_df in labels_boats_merged.groupby(["site_year", "Begin File"]):

    # if site_year == "BSM_2017":
    #     continue
    num_boats = (group_df["Type"] == "Boat").sum()
    num_ambients = (group_df["Type"] == "Ambient").sum()
    print(f"Processing {site_year} {audio_file}, {len(group_df)} passages, {num_boats} boats, {num_ambients} ambients")
    file_of_interest = audio_file.replace(".wav", "")
    


    wavs_folder = wavs_folder_dict[site_year]

    audio_file_path = wavs_folder + audio_file

    if not os.path.exists(audio_file_path):
        print(f"Audio file {audio_file_path} does not exist. Skipping...")
        continue
    # print(audio_start_time)
    long_audio, sample_rate, original_sr = spect_generator.load_audio(audio_file_path) 


    pipeline_output_path =f"./outputs/{site_year}/{file_of_interest}.csv"
    if not os.path.exists(pipeline_output_path):
        print(f"Outputs file {pipeline_output_path} does not exist. Skipping...")
        continue
    results_df = pd.read_csv(pipeline_output_path)
    raven_df = translate_to_raven(results_df, sample_rate)


    for i, row in group_df.iterrows():
        print(row["Type"])
        passage_id = row["Selection"]
        passage_start_s = row["Begin Time (s)"]
        passage_end_s = row["End Time (s)"]
        context = row["Type"]
        boat_labeling_file = row["source_label_file"]

        cur_snippet_dir = f"../data/evaluation_snippets/v2/{site_year}/{context}/"
        os.makedirs(cur_snippet_dir, exist_ok=True)

        duration_s = passage_end_s - passage_start_s

        #Only take 1 snippet for ambient
        max_snippets = 3 if context == "Boat" else 1

        snippets = create_passage_snippets(passage_start_s, passage_end_s, max_snippets=max_snippets)
        print(f"passage is {duration_s}s long, {len(snippets)} snippets")
        print(snippets)
        for i, (start_s, end_s) in enumerate(snippets):
            passage_df, snippet_filename, calls_detected = create_snippet_df(raven_df, audio_file, start_s, end_s, boat_labeling_file, passage_id)
            passage_df["context"] = context

            print("calls detected", calls_detected)
            clip_and_save_audio(long_audio, sample_rate, start_s, end_s, snippet_filename, cur_snippet_dir)

            pipeline_output_file = snippet_filename.replace(".wav", ".selections.txt")
            

            passage_timings = ["Start", "Middle", "End"]

            if start_s == passage_start_s and end_s == passage_end_s:
                passage_timing = "Full_Passage"
            else:
                # If we have 2 snippets, the first is "Start" and the second is "End"
                if len(snippets) == 2:
                    passage_timing = "Start" if i == 0 else "End"
                else:
                    passage_timing = passage_timings[i]
            
            

            snippets_df = snippets_list.append({
                "site_year": site_year,
                "snippet_filename": snippet_filename,
                "pipeline_output_file": pipeline_output_file,
                "snippet_duration_s": end_s - start_s,
                "context": context,
                "ECHO_calls": calls_detected["ECHO"],
                "HFPC_calls": calls_detected["HFPC"],
                "BBPC_calls": calls_detected["BBPC"],
                "Whistle_calls": calls_detected["Whistle"],
                "source_wav_file": audio_file,
                "source_label_file": boat_labeling_file,
                "snippet_start_s": start_s,
                "snippet_end_s": end_s,
                
                "snippet_passage_timing": passage_timing,

                "passage_start_s": passage_start_s,
                "passage_end_s": passage_end_s,
                "source_label_passage_id": passage_id,
                "unique_passage_id": f"{audio_file}_{passage_id}",
                
                "labeled": False
            })

            passage_df.to_csv(cur_snippet_dir + pipeline_output_file, sep="\t", index=False)

    # break  # Remove or adjust this if you want to process more than one file group

snippets_df = pd.DataFrame(snippets_list)

Processing BSM_2017 201359382.170724093002.wav, 2 passages, 1 boats, 1 ambients
Boat
passage is 516s long, 3 snippets
[(1035, 1155), (1233, 1353), (1431, 1551)]
calls detected {'ECHO': 0, 'HFPC': 0, 'BBPC': 0, 'Whistle': 3}
Saved 120.00s audio clip to: c:\Users\Admin\Desktop\Emmanuel\beluga-call-pipeline\data\evaluation_snippets\v2\BSM_2017\Boat\201359382.170724094717.snippet.wav
Clip shape: (23040000,), Sample rate: 192000 Hz
calls detected {'ECHO': 0, 'HFPC': 0, 'BBPC': 0, 'Whistle': 0}
Saved 120.00s audio clip to: c:\Users\Admin\Desktop\Emmanuel\beluga-call-pipeline\data\evaluation_snippets\v2\BSM_2017\Boat\201359382.170724095035.snippet.wav
Clip shape: (23040000,), Sample rate: 192000 Hz
calls detected {'ECHO': 0, 'HFPC': 0, 'BBPC': 0, 'Whistle': 0}
Saved 120.00s audio clip to: c:\Users\Admin\Desktop\Emmanuel\beluga-call-pipeline\data\evaluation_snippets\v2\BSM_2017\Boat\201359382.170724095353.snippet.wav
Clip shape: (23040000,), Sample rate: 192000 Hz
Ambient
passage is 912s long,

In [ ]:
num_boat_unique_passages = snippets_df[snippets_df["context"] == "Boat"]["source_label_passage_id"].nunique()
print(f"Number of unique 'Boat' passages: {num_boat_unique_passages}")

Number of unique 'Boat' passages: 6


In [28]:
snippets_df["unique_passage_id"] = snippets_df["source_label_passage_id"].astype(str) + "_" + snippets_df["source_label_file"]
num_boat_unique_passages = snippets_df[snippets_df["context"] == "Boat"]["unique_passage_id"].nunique()
print(f"Number of unique 'Boat' passages: {num_boat_unique_passages}")

Number of unique 'Boat' passages: 45


In [31]:
snippets_df["snippet_duration_s"] = snippets_df["snippet_end_s"] - snippets_df["snippet_start_s"]
snippets_df[snippets_df["context"] == "Boat"]["snippet_duration_s"].sum()/60

197.83333333333334

In [35]:
snippets_df.to_csv(f"../data/evaluation_snippets/v2/snippets_df.csv", index=False)

raven_df = translate_to_raven(results_df, sample_rate)
